# 🔬 OptoBind-AI — Pipeline de Design De Novo de Biosenseurs Optiques

Ce notebook permet d'exécuter l'intégralité du pipeline **OptoBind-AI** sur un **GPU NVIDIA gratuit dans le Cloud (Google Colab)** sans avoir besoin de carte graphique sur votre propre ordinateur !

---

### Étape 1 : Activer le GPU NVIDIA gratuit
1. Dans le menu en haut, cliquez sur **Exécution** > **Modifier le type d'exécution**.
2. Sous **Accélérateur matériel**, choisissez **T4 GPU** (gratuit).
3. Cliquez sur **Enregistrer**.

Vérifions la présence du GPU :

In [ ]:
!nvidia-smi

### Étape 2 : Cloner votre projet depuis GitHub

In [ ]:
%cd /content
!rm -rf Optobind-AI
!git clone https://github.com/Lorenzo-colao/Optobind-AI.git
%cd /content/Optobind-AI

### Étape 3 : Installer les dépendances bio-informatiques et télécharger les poids

In [ ]:
# Installation rapide des dépendances compatibles Colab
!pip install --quiet biopython rdkit openpyxl pandas pyyaml tqdm
!pip install --quiet chai-lab fair-esm

# Clonage des moteurs de calcul
!git clone https://github.com/RosettaCommons/RFdiffusion.git
!git clone https://github.com/dauparas/ProteinMPNN.git

# Installation du module SE3Transformer interne à RFdiffusion
!cd RFdiffusion/env/SE3Transformer && pip install --no-cache-dir -r requirements.txt && python setup.py install
!cd RFdiffusion && pip install -e .

# Téléchargement des poids
!mkdir -p weights
!wget -c -P weights/ http://files.ipd.uw.edu/pub/RF-All-Atom/weights/RFDiffusionAA_paper_weights.pt
!wget -c -O weights/ProteinMPNN_v_48_020.pt https://raw.githubusercontent.com/dauparas/ProteinMPNN/main/vanilla_model_weights/v_48_020.pt

print("\n🎉 TOUTES LES DÉPENDANCES ET POIDS SONT INSTALLÉS !")

### Étape 4 : Lancer le pipeline OptoBind-AI sur GPU !

In [ ]:
!python orchestrator.py \
  --target inputs/targets/zn2+_site.pdb \
  --ligand inputs/ligands/DFHBI-1T.sdf

### Étape 5 : Télécharger votre rapport Excel de validation

In [ ]:
import glob
from google.colab import files

# Recherche du rapport généré
reports = glob.glob('outputs/**/reports/report.xlsx', recursive=True)
if reports:
    print(f"Téléchargement de : {reports[-1]}")
    files.download(reports[-1])
else:
    print("Aucun rapport trouvé.")